# Aula 07 — Mini-Projeto — Aula 07 (coleta de dataset real via API)

Este notebook é o execício da aula 7 da disciplina Análise e Ciências de Dados do curso de Ciências de Dados do IFRN para desenvolver um script com a finalidade de carregar dados de fontes distintas (json, API), gravar em um banco de dados SQLite, escrever uma consulta SQL, exportar para outros formatos, demonstrar os processos de verificação érica e legal que pode ser seguido para fazer scrpaing e gerar um gráfico a partir de dados de geo localização.

Escolha uma API pública real (pode ser o IBGE, o TSE, ou qualquer outra API aberta e gratuita de sua preferência) e construa um notebook que colete um dataset real a partir dela, aplicando os conceitos da Aula 07.

**Requisitos mínimos:**

- Pelo menos uma requisição GET com requests, com tratamento de status/erro (try/except + raise_for_status()).
- Paginação, se a API escolhida limitar o tamanho da resposta (se a API não paginar, explique por escrito, no próprio notebook, por que não foi necessária).
- Salvar o resultado em JSON e CSV, com um bloco de proveniência (fonte, endpoint, data da coleta, licença, se conhecida).
- Um parágrafo final (em uma célula Markdown) descrevendo: qual API foi usada, quantos registros foram coletados e uma observação sobre os dados.

**Sugestões de APIs públicas e gratuitas (sem necessidade de cartão de crédito):**

* IBGE — Localidades (servicodados.ibge.gov.br) ou Agregados/SIDRA
* TSE — Portal de Dados Abertos (dadosabertos.tse.jus.br)
* ViaCEP (viacep.com.br) — endereços por CEP, sem autenticação
* BrasilAPI (brasilapi.com.br) — feriados, bancos, CNPJ, tabela FIPE, sem autenticação
* OpenWeather (openweathermap.org) — previsão do tempo, exige API Key gratuita (boa escolha para praticar autenticação)
* PokéAPI (pokeapi.co) — catálogo grande, ótimo para praticar paginação por cursor

O que entregar: o notebook (.ipynb) já executado, com as saídas visíveis, mais os arquivos JSON/CSV gerados — todos compactados em um único .zip.

## 1. Preparação de ambiente

Configurar o ambiente de trabalho, importar bibliotecas necessárias, criar a pasta "dados" onde será salvo os dados da resposta da API.

In [1]:
import warnings
warnings.filterwarnings('ignore', message='Unverified HTTPS request')

import requests
import json
from pathlib import Path
from urllib.parse import urlparse
from datetime import datetime
import pandas as pd

# criar a pasta onde será salvo os dados baixados
DADOS = Path("dados")
DADOS.mkdir(exist_ok=True)

print("Versão do requests:", requests.__version__)

Versão do requests: 2.34.2


## 2. Coleta de dados via API

Função para coletar os dados da API escolhida, verificar status da resposta e tratar possíveis erros.

In [2]:
def fazer_requisicao(url, params=None, timeout=10):
    """Faz requisição GET e retorna JSON."""

    try:
        resposta = requests.get(
            url,
            params=params,
            timeout=timeout,
            verify=False
        )
        resposta.raise_for_status()
        print("\nStatus code:", resposta.status_code)
        print("Content-Type:", resposta.headers.get("Content-Type"))
        return resposta.json()
    except requests.exceptions.RequestException as erro:
        raise RuntimeError(f"Erro na requisição GET para {url}: {erro}") from erro
    except ValueError as erro:
        raise RuntimeError("A resposta da API não está em formato JSON válido.") from erro

## 3. Salva dados

Salvar os dados em JSON e CSV na pasta "dados" com informações de proveniência como metadados (fonte, endpoint, data da coleta, licença, se conhecida).

In [3]:
def salvar_dados(proveniencia, dados):
    """Salva dados em JSON e CSV com proveniência."""
    base = DADOS / Path(proveniencia["nome_arquivo"]).stem
    
    # Salvar JSON
    with (base.with_suffix(".json")).open("w", encoding="utf-8") as f:
        json.dump({"proveniencia": proveniencia, "dados": dados}, f, ensure_ascii=False, indent=2)
    
    # Salvar CSV
    df = pd.json_normalize(dados)
    with (base.with_suffix(".csv")).open("w", encoding="utf-8", newline="") as f:
        for k, v in proveniencia.items():
            f.write(f"# {k}: {v}\n")
        df.to_csv(f, index=False)
    
    print(f"✓ Salvo: {base.with_suffix('.json')} e {base.with_suffix('.csv')}")
    return df

## 4. Requisição simples sem paginação

Consultar a url (endpoint) do IBGE para coletar os dados de municípios do estado do RN, usando a biblioteca "requests". Salvar o resultado em JSON e CSV, com um bloco de proveniência (url, documentacao, fonte, licença, data da coleta, nome_arquivo, quantidade de registro).

In [4]:
# EXEMPLO 1: IBGE Municípios (sem paginação)
uf = "RN"
url = f"https://servicodados.ibge.gov.br/api/v1/localidades/estados/{uf}/municipios"
params = {}
timeout=10

respostaJson = fazer_requisicao(url, params, timeout)

print(f"\nTotal de municípios recebidos: {len(respostaJson)}")
print("Exemplo (primeiro registro):")
print(json.dumps(respostaJson[0], ensure_ascii=False, indent=2))

proveniencia = {
    "url": url,
    "api_doc": "https://servicodados.ibge.gov.br/api/docs/localidades#api-Municipios-estadosUFMunicipiosGet",
    "fonte": "Instituto Brasileiro de Geografia e Estatística (IBGE)",
    "licenca": "Creative Commons Atribuição - CC BY",
    "data_coleta": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "nome_arquivo": f"municipios_{uf}",
    "qtd_registros": len(respostaJson)
}

# Salvar os dados e exibir o DataFrame
df = salvar_dados(proveniencia, respostaJson)
df.head()


Status code: 200
Content-Type: application/json; charset=utf-8

Total de municípios recebidos: 167
Exemplo (primeiro registro):
{
  "id": 2400109,
  "nome": "Acari",
  "microrregiao": {
    "id": 24012,
    "nome": "Seridó Oriental",
    "mesorregiao": {
      "id": 2402,
      "nome": "Central Potiguar",
      "UF": {
        "id": 24,
        "sigla": "RN",
        "nome": "Rio Grande do Norte",
        "regiao": {
          "id": 2,
          "sigla": "NE",
          "nome": "Nordeste"
        }
      }
    }
  },
  "regiao-imediata": {
    "id": 240008,
    "nome": "Currais Novos",
    "regiao-intermediaria": {
      "id": 2402,
      "nome": "Caicó",
      "UF": {
        "id": 24,
        "sigla": "RN",
        "nome": "Rio Grande do Norte",
        "regiao": {
          "id": 2,
          "sigla": "NE",
          "nome": "Nordeste"
        }
      }
    }
  }
}
✓ Salvo: dados/municipios_RN.json e dados/municipios_RN.csv


,id,nome,microrregiao.id,microrregiao.nome,microrregiao.mesorregiao.id,microrregiao.mesorregiao.nome,microrregiao.mesorregiao.UF.id,microrregiao.mesorregiao.UF.sigla,microrregiao.mesorregiao.UF.nome,microrregiao.mesorregiao.UF.regiao.id,...,regiao-imediata.id,regiao-imediata.nome,regiao-imediata.regiao-intermediaria.id,regiao-imediata.regiao-intermediaria.nome,regiao-imediata.regiao-intermediaria.UF.id,regiao-imediata.regiao-intermediaria.UF.sigla,regiao-imediata.regiao-intermediaria.UF.nome,regiao-imediata.regiao-intermediaria.UF.regiao.id,regiao-imediata.regiao-intermediaria.UF.regiao.sigla,regiao-imediata.regiao-intermediaria.UF.regiao.nome
0,2400109,Acari,24012,Seridó Oriental,2402,Central Potiguar,24,RN,Rio Grande do Norte,2,...,240008,Currais Novos,2402,Caicó,24,RN,Rio Grande do Norte,2,NE,Nordeste
1,2400208,Assú,24004,Vale do Açu,2401,Oeste Potiguar,24,RN,Rio Grande do Norte,2,...,240011,Açu,2403,Mossoró,24,RN,Rio Grande do Norte,2,NE,Nordeste
2,2400307,Afonso Bezerra,24009,Angicos,2402,Central Potiguar,24,RN,Rio Grande do Norte,2,...,240011,Açu,2403,Mossoró,24,RN,Rio Grande do Norte,2,NE,Nordeste
3,2400406,Água Nova,24005,Serra de São Miguel,2401,Oeste Potiguar,24,RN,Rio Grande do Norte,2,...,240010,Pau dos Ferros,2403,Mossoró,24,RN,Rio Grande do Norte,2,NE,Nordeste
4,2400505,Alexandria,24006,Pau dos Ferros,2401,Oeste Potiguar,24,RN,Rio Grande do Norte,2,...,240010,Pau dos Ferros,2403,Mossoró,24,RN,Rio Grande do Norte,2,NE,Nordeste


## 5. Requisição com paginação

Consultar a url (endpoint) do TSE para coletar os dados de resultados de eleições com paginação, usando a biblioteca "requests".

In [5]:
# EXEMPLO 2: Paginação até pegar tudo
url = f"https://dadosabertos.tse.jus.br/api/3/action/package_search"

termo = "eleições"
start = 0
rows = 5

params = {
    "q": termo,
    "start": start,
    "rows": rows
}
timeout=10
paginas = []

while True:
    respostaJson = fazer_requisicao(url, params, timeout)
    pagina = respostaJson["result"]
    paginas.extend(pagina["results"])
    print(f"start={start}: +{len(pagina['results'])} resultados (total até agora: {len(paginas)}/{pagina['count']})")
    if start + rows >= pagina["count"]:
        break
    start += rows

print()
print("Conjuntos de dados coletados:")
for p in paginas:
    print(" -", p["name"])

proveniencia = {
    "url": url,
    "api_doc": "https://dadosabertos.tse.jus.br/",
    "fonte": "Tribunal Superior Eleitoral (TSE)",
    "licenca": "Creative Commons Atribuição - CC BY",
    "data_coleta": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "nome_arquivo": "eleicoes",
    "qtd_registros": len(paginas)
}

# Salvar os dados e exibir o DataFrame
df = salvar_dados(proveniencia, paginas)
df.head()


Status code: 200
Content-Type: application/json;charset=utf-8
start=0: +5 resultados (total até agora: 5/9)

Status code: 200
Content-Type: application/json;charset=utf-8
start=5: +5 resultados (total até agora: 10/9)

Conjuntos de dados coletados:
 - resultados-eleicoes-suplementares-2024
 - resultados-eleicoes-suplementares-2022
 - resultados-2020-suplementares
 - resultados-1998
 - resultados-1996
 - resultados-eleicoes-suplementares-2024
 - resultados-eleicoes-suplementares-2022
 - resultados-2020-suplementares
 - resultados-1998
 - resultados-1996
✓ Salvo: dados/eleicoes.json e dados/eleicoes.csv


,author,author_email,creator_user_id,id,isopen,license_id,license_title,maintainer,maintainer_email,metadata_created,...,organization.name,organization.title,organization.type,organization.description,organization.image_url,organization.created,organization.is_organization,organization.approval_status,organization.state,license_url
0,,,41b5c78a-7e64-4e18-bbed-2b4459b62ebb,3491a00c-a661-43d7-b5be-b8952a5d80d7,False,,,,,2025-04-04T20:07:25.605478,...,tse-agel,TSE/AGEL,organization,Assessoria de Gestão Eleitoral - AGEL,,2021-09-08T15:14:51.108227,True,approved,active,NaN
1,,,41b5c78a-7e64-4e18-bbed-2b4459b62ebb,a1c1f622-353f-4e65-9391-9f5dc6c1e437,False,,,,,2026-06-18T22:40:53.439445,...,tse-agel,TSE/AGEL,organization,Assessoria de Gestão Eleitoral - AGEL,,2021-09-08T15:14:51.108227,True,approved,active,NaN
2,,,41b5c78a-7e64-4e18-bbed-2b4459b62ebb,67ed33ec-0ff5-40c2-ab5d-1812a375085a,False,,,,,2023-08-10T23:02:22.776713,...,tse-agel,TSE/AGEL,organization,Assessoria de Gestão Eleitoral - AGEL,,2021-09-08T15:14:51.108227,True,approved,active,NaN
3,,,d53ef805-4d92-4e45-87e5-cc0f2d7caf98,fc3c2692-abbd-4ec5-9ef3-ca4dffc0f161,True,cc-by,Creative Commons Attribution,,,2021-08-27T17:24:08.378014,...,tse-agel,TSE/AGEL,organization,Assessoria de Gestão Eleitoral - AGEL,,2021-09-08T15:14:51.108227,True,approved,active,http://www.opendefinition.org/licenses/cc-by
4,,,d53ef805-4d92-4e45-87e5-cc0f2d7caf98,7885a4eb-6e37-4951-9d44-52362296ce19,True,cc-by,Creative Commons Attribution,,,2021-08-27T16:57:52.922925,...,tse-agel,TSE/AGEL,organization,Assessoria de Gestão Eleitoral - AGEL,,2021-09-08T15:14:51.108227,True,approved,active,http://www.opendefinition.org/licenses/cc-by


## 6. Observações finais

Foi usado a API de municipios do IBGE (apenas os dados do municipio do estado do RN) com total de **167** municipios sem necessidade de paginação porque a API do IBGE retorna todos os municípios de uma vez. E a API do TSE (resultados de eleições) com paginação, com total de **10** registros coletados, sendo necessário fazer múltiplas requisições para obter todos os resultados.

A API do IBGE não possui paginação, pois retorna todos os municípios de uma vez. Já a API do TSE possui paginação, e foi necessário fazer múltiplas requisições para obter todos os resultados.

Observações:
- É importante sempre verificar a documentação da API para entender como os dados são estruturados e quais informações estão disponíveis. Os dados coletados do IBGE são atualizados periodicamente, e podem sofrer alterações ao longo do tempo. Portanto, é importante sempre verificar a data da coleta e a versão da API utilizada.
- O JSON retornado pela API pode vir aninhado e com diferentes níveis de profundidade, o que pode dificultar a análise dos dados. Por isso, é importante sempre verificar a estrutura do JSON retornado, entender como os dados estão organizados e, se necessário, realizar o pré-processamento, como por exemplo, usar a função `json_normalize()` do pandas para transformar os dados em um formato tabular, facilitando a análise e manipulação dos dados.